# Оптимизация водохозяйственной системы бассейна Зарафшана
*Учебное пособие по многокритериальной оптимизации*
**Авторы:** Dr. Tobias Siegfried, Nicolas Lazaro, Dr. Beatrice Marti (hydrosolutions GmbH)  
**Лицензия:** CC BY 4.0

---

::: {.callout-note title="Цели обучения"}
После изучения пособия вы сможете:

1. **Сформировать** физическое понимание компромиссов при распределении воды, вручную изменяя параметры и наблюдая реакцию системы.
2. **Сформулировать** задачу оптимизации: перечислить переменные решения и выбрать, какие фиксировать, а какие настраивать.
3. **Объяснить** оптимальность по Парето и отсутствие единственного «лучшего» решения при конфликтующих целях.
4. **Настроить** параметры оптимизации: границы, ограничения, размер популяции и число поколений.
5. **Запустить** NSGA-II и оценить сходимость по гиперобъёму.
6. **Анализировать** фронты Парето с помощью попарных диаграмм рассеяния, параллельных координат и сравнения характерных решений.
7. **Обсудить** способы уменьшения сложности и их физическое обоснование.
8. **Интерпретировать** оптимизированные параметры, чтобы понять, что оптимизатор «узнал» о системе.
:::

::: {.callout-warning title="Необходимые знания"}
Это пособие дополняет *Модель водохозяйственной системы бассейна Зарафшана: учебное пособие по моделированию*. Предполагается, что вы прошли его и знакомы со следующими понятиями:

- **Модель системы ZRB**: суточная модель узлов и связей, включающая источники, водохранилища, распределители, потребителей, транзитные узлы, конечные узлы и участки русла.
- **Водный баланс**: приток, потребление, потери, изменение запасов.
- **Правило выпуска SLOP** (Storage Level Operating Policy) для водохранилищ, связывающее выпуск с уровнем запасов.
- **Потери при транспортировке по каналам**: фильтрационные, испарительные и эксплуатационные.
- **Базовый вариант с постоянными настроенными долями**: одна фиксированная доля, взвешенная по спросу, для каждой подходящей ветви распределителя, применяемая при суточном расчёте.
- Основы Python (функции, словари, списковые включения), pandas, matplotlib/plotly.
:::

::: {.callout-note title="Единицы и условные обозначения" collapse="true"}
| Символ / единица | Значение |
|:-----------------------------------|:-----------------------------------|
| m³/s | Расход воды (кубические метры в секунду) |
| m³/day | Объём за суточный шаг = расход × 86,400 секунд в сутки |
| Mm³ | Миллион кубических метров (10⁶ m³) — для ёмкости водохранилищ и годовых объёмов |
| Bm³ | Миллиард кубических метров (10⁹ m³) — для суммарного годового притока |
| Суточный шаг | Расчёт за 2017–2022 годы содержит 2191 суточный шаг |

Исходные расходы загружаются в m³/s и внутри модели переводятся в объёмы за суточный шаг. Для удобства чтения накопленные объёмы в таблицах приведены в Mm³ или Bm³.
:::

# Введение: от моделирования к оптимизации {#sec-introduction}

## Зачем нужна оптимизация? {#sec-why-optimise}

В пособии по моделированию мы создавали разные варианты водохозяйственной системы Зарафшана, сравнивая простые правила распределения воды. Самое простое правило — пропорциональное распределение: каждый гидроузел делит поступающий расход между выходящими участками в фиксированных долях, даже когда это приводит к непроизводительным сбросам или игнорирует спрос ниже по течению.

Затем базовая модель с постоянными настроенными долями задала каждому подходящему распределителю HydroWorks одну фиксированную, взвешенную по спросу долю для каждой нижележащей ветви, применяемую ежедневно. Это существенное улучшение по сравнению с равными долями: правило учитывает *величину* спроса ниже по течению. Однако оно не позволяет согласовать конкурирующие цели.

Постоянные настроенные доли вычислены по суммарному спросу за 6 лет. Они не учитывают сезонные изменения и то, что увеличение подачи сельскому хозяйству может уменьшить поступление воды на Навоийскую электростанцию или ухудшить равномерность распределения между районами. Это ответ на вопрос «как распределять воду в среднем?», но не на вопрос «каково наилучшее достижимое распределение, если одновременно важны несколько, возможно конкурирующих, целей?».

Это пособие отвечает на второй вопрос. Мы переходим от **моделирования** — оценки заданного правила управления — к **оптимизации**, то есть поиску лучшего правила. Среди всех допустимых настроек мы ищем те, которые минимизируют дефицит в сельском хозяйстве, сохраняя надёжность водоснабжения электростанции, справедливое распределение между пользователями и достаточно плавный режим работы для практического применения.

Мы берём базовую модель с постоянными настроенными долями по двум причинам. Во-первых, её логика распределения понятна: решения распределителей задаются структурой правил, а не произвольными свободными долями в каждой ветви на каждом шаге (см. @sec-decision-space). Это полезно для обучения: можно понять физический смысл каждого правила и проследить его влияние на систему.

Во-вторых, эта модель достаточно быстра для обучения, но сохраняет реалистичные сложности оптимизации. Каждая оценка включает моделирование 2191 суточного шага и расчёт четырёх критериев. Текущая задача содержит небольшое число параметров правил распределителей и месячно-циклические параметры водохранилищ. Это позволяет обсуждать размерность, сходимость и интерпретируемость, не меняя суточный шаг модели.

## Идея Парето: почему нет единственного «лучшего» решения {#sec-pareto-idea}

Предположим, можно уменьшить общий сельскохозяйственный дефицит на 100 Mm³, но только ценой увеличения нехватки воды для охлаждения электростанции на 20 Mm³. Стоит ли это делать? Что, если кто-то считает продовольственную безопасность важнее надёжности промышленности — или наоборот? При конфликтующих целях единственного оптимального ответа нет.

Вместо этого существует *множество* наилучших достижимых компромиссов — **фронт Парето**. Основная идея показана на @fig-pareto-idea.

На фронте Парето улучшение одного критерия обязательно ухудшает другой. Каждая точка отражает свой выбор относительной важности целей. Оптимизатор не выбирает за вас: он показывает всё множество достижимых компромиссов, предоставляя информацию для обоснованных и согласованных решений.


<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>


In [ ]:
# @title Подготовка: сначала выполните эту ячейку
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Рисунок: компромисс двух критериев
#| label: fig-pareto-idea
#| fig-cap: "Схема компромисса двух критериев. Каждая точка — отдельная конфигурация параметров. Синие точки фронта Парето недоминируемы: ни одна другая конфигурация не лучше сразу по обоим критериям. Серые точки доминируются: хотя бы одно решение фронта лучше по каждому критерию. Затенённая область недостижима."
from zarafshan_taqsim import figures, plot_pareto_idea, tables

figures.reset()
tables.reset()
fig = plot_pareto_idea()
figures.caption(
    fig,
    "Схема компромисса двух критериев. Каждая точка — отдельная конфигурация параметров. Синие точки фронта Парето недоминируемы: ни одна другая конфигурация не лучше сразу по обоим критериям. Серые точки доминируются: хотя бы одно решение фронта лучше по каждому критерию. Затенённая область недостижима.",
    label="pareto-idea",
)

Мы дадим формальное определение в @sec-pareto-formal после практического знакомства в @sec-building-intuition.

## Планирование будущего {#sec-planning}

Бассейн Зарафшана испытывает несколько взаимосвязанных нагрузок. Рост населения Самарканда повышает спрос на воду для орошения и коммунального хозяйства. Отступание ледников в верховьях уменьшает летний сток, от которого зависят водохранилища. Старение каналов снижает эффективность транспортировки. Эти процессы усиливают друг друга: растущему населению нужно больше воды из сокращающихся ресурсов, подаваемых через ухудшающуюся инфраструктуру.

Подход к оптимизации в этом пособии не ограничен историческим рядом 2017–2022 годов. TaqSim [@taqsim] — универсальная система моделирования с регистрацией событий: она может принимать любой гидрологический ряд, включая искусственные засухи, сценарии отступания ледников или роста спроса, и рассчитывать соответствующее поведение системы. Затем модуль оптимизации TaqSim ищет распределения, оптимальные по Парето, для изменённых условий. Всю последовательность «задать сценарии → моделировать → оптимизировать → анализировать фронт Парето» можно повторить с другими исходными данными без изменения кода.

Это открывает возможности **сценарного планирования**. Управляющий водными ресурсами может спросить: «Если летний приток уменьшится на 30% из-за отступания ледников, как сместится фронт Парето? Какие стратегии распределения сохранят работоспособность, а какие перестанут работать?». Запуская оптимизатор для нескольких правдоподобных вариантов будущего — засушливого ряда, быстрого роста спроса, программы реконструкции каналов — и сравнивая фронты Парето, можно выявить **устойчивые варианты распределения**, приемлемые в разных условиях. Это полезнее правил, оптимизированных лишь для одного исторического ряда, который может больше не повториться.

Для долгосрочного бассейнового планирования важно, что фронт Парето — не неизменный ответ, а карта, меняющаяся вместе с условиями. Инструменты пособия — TaqSim и многокритериальная оптимизация — позволяют быстро и систематически пересчитывать эту карту и корректировать управление по мере изменения гидрологических и социально-экономических условий.

## Структура пособия {#sec-roadmap}

Десять разделов последовательно развивают тему:

1. **Введение** (этот раздел): мотивация и идея Парето.
2. **Подготовка и повторение базового варианта** (@sec-baseline): настройка среды и исходный ориентир для сравнения.
3. **Развитие интуиции** (@sec-building-intuition): ручное изменение параметров для знакомства с компромиссами.
4. **Пространство решений** (@sec-decision-space): настраиваемые параметры распределителей и водохранилищ.
5. **Пространство критериев** (@sec-objective-space): 9 возможных критериев, выбор 4 и оценка базового варианта.
6. **Понятия оптимизации** (@sec-optimisation-concepts): теория Парето, NSGA-II и организация оптимизатора.
7. **Результаты оптимизации** (@sec-optimisation-results): запуск NSGA-II и анализ фронта Парето.
8. **Обсуждение** (@sec-discussion): что оптимизация показывает и чего не показывает.
9. **Выводы** (@sec-conclusions): основные результаты и практическое значение.
10. **Сборник упражнений** (@sec-exercises): все упражнения со ссылками на разделы.

::: {.callout-important title="Главное — введение"}
- Базовый вариант с постоянными настроенными долями — полезное правило, но он не согласует конкурирующие цели.
- При конфликтующих целях нет единственного оптимального решения: существует фронт Парето наилучших достижимых компромиссов.
- Оптимизация показывает компромиссы; решение выбирают на основе *человеческих ценностей*.
:::

# Подготовка и повторение базового варианта {#sec-baseline}

## Подготовка {#sec-setup}


In [ ]:
# @title Подготовка · код
#| label: setup
#| echo: true
import logging

import matplotlib.pyplot as plt

from zarafshan_taqsim import (
    N_TIMESTEPS,
    apply_static_demand_splitting,
    baseline_district_summary,
    baseline_objective_table,
    build_objectives,
    compute_baseline,
    compute_reference_point,
    create_zrb_system,
    decision_table_from_schema,
    figures,
    tables,
    front_arrays,
    kattakurgan_policy_variants,
    narpay_equity_variants,
    objective_catalog_table,
    parameter_schema_summary,
    plot_convergence,
    plot_narpay_equity,
    plot_parallel_coordinates,
    plot_pareto_concept,
    plot_pareto_3d,
    plot_pareto_scatter,
    plot_release_sensitivity,
    plot_representative_district_deficits,
    plot_representative_sink_flows,
    plot_selected_parameter_profiles,
    plot_tunable_parameter_map,
    release_comparison_table,
    representative_indices,
    representative_table,
    run_coursebook_optimization,
    simulate_representatives,
    visualize_network,
)

# Constants
TIMESTEPS = N_TIMESTEPS
SEED = 42

# Plotting defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

logging.basicConfig(level=logging.WARNING)

## Базовый вариант с постоянными настроенными долями {#sec-static-baseline}

Базовая система разработана в пособии по моделированию. Она использует `create_zrb_system(use_canal_losses=True)`, включая реалистичные потери в каналах, вместе с `apply_static_demand_splitting()`. Эта функция рассчитывает для каждой подходящей ветви распределителя одну фиксированную долю по накопленному спросу ниже по течению. Модель по-прежнему работает с суточным шагом; доли просто повторяются каждый день.


In [ ]:
# @title Базовый вариант с постоянными настроенными долями · код
#| label: baseline-setup
baseline = create_zrb_system(use_canal_losses=True)
apply_static_demand_splitting(baseline, verbose=False)
baseline.validate()
baseline.simulate(TIMESTEPS)

На @fig-network-map показана географическая схема водохозяйственной системы Зарафшана. Вода поступает в верховьях через узел Source и проходит через сеть распределителей HydroWorks (оранжевые), подающих воду оросительным районам (красные), водохранилищам (зелёные) и конечным узлам (серые). Два водохранилища — Каттакурганское и Акдарьинское — обеспечивают накопление и регулируемые выпуски. Электростанция Powerplant (фиолетовая) получает охлаждающую воду от HW_Karmana. Понимание этой схемы необходимо для интерпретации оптимизации: решения выше по течению распространяются по сети и влияют на каждый нижележащий узел.


In [ ]:
# @title Рисунок: географическая схема системы Зарафшана
#| label: fig-network-map
#| fig-cap: "Географическая схема водохозяйственной системы Зарафшана. Положения узлов соответствуют UTM Zone 42N. Цвета: синий — Source (источник), зелёный — Storage (водохранилище), оранжевый — Splitter (гидроузел), красный — Demand (оросительный район), фиолетовый — PassThrough (электростанция), серый — Sink (конечный узел)."
fig, _ = visualize_network(baseline, show_reaches=False, figsize=(18, 12))
figures.caption(
    fig,
    "Географическая схема водохозяйственной системы Зарафшана. Положения узлов соответствуют UTM Zone 42N. Цвета: синий — Source (источник), зелёный — Storage (водохранилище), оранжевый — Splitter (гидроузел), красный — Demand (оросительный район), фиолетовый — PassThrough (электростанция), серый — Sink (конечный узел).",
    label="network-map",
)

In [ ]:
# @title Таблица: удовлетворение спроса по районам
#| label: tbl-baseline-districts
#| tbl-cap: "Удовлетворение спроса районов в базовом варианте с постоянными настроенными долями. Дефицит — накопленный неудовлетворённый спрос за 2191 суточный шаг."
tables.caption(
    baseline_district_summary(baseline, TIMESTEPS).style.hide(axis="index"),
    "Удовлетворение спроса районов в базовом варианте с постоянными настроенными долями. Дефицит — накопленный неудовлетворённый спрос за 2191 суточный шаг.",
    label="baseline-districts",
)

Базовый вариант служит ориентиром во всём пособии. Оптимизатор должен показать, что способен улучшить результат, и какой ценой для других критериев. В @tbl-baseline-districts показано удовлетворение спроса по районам — важный показатель работы базовой системы. Подробности настройки системы приведены в главе пособия о моделировании водохозяйственной системы.

::: {.callout-important title="Главное — базовый вариант"}
- Для каждой подходящей ветви распределителя задана одна постоянная доля, взвешенная по спросу и применяемая ежедневно.
- Потери в каналах включены, поэтому исходный вариант реалистичен.
- Некоторые районы, особенно самые нижние по течению, испытывают больший дефицит.
:::

# Развитие интуиции: сначала попробуйте вручную {#sec-building-intuition}

Прежде чем поручить оптимизатору множество сочетаний параметров, изменим параметры вручную и посмотрим на последствия. Это развивает физическое понимание и делает фронт Парето содержательным, а не абстрактным. Студенты, столкнувшиеся с компромиссами вручную, увидят в результате оптимизатора количественное выражение собственного опыта.

## Что произойдёт при изменении правил водохранилища? {#sec-release-sensitivity}

Базовый вариант использует для Каттакурганского водохранилища SLOP (Storage Level Operating Policy) с четырьмя параметрами: целевым расходом выпуска `vr`, двумя границами зон запасов `v1` (верх буферной зоны) и `v2` (верх зоны сохранения запасов), а также `buffer_coef`, определяющим интенсивность выпуска в буферной зоне между мёртвым объёмом и `v1`. Что произойдёт при принципиально иной стратегии управления?

Сравним три разных стратегии. **Консервативная** задаёт малый расход выпуска, высокую границу зоны сохранения и минимальный буферный коэффициент: вода активно удерживается, выпуск происходит лишь при больших запасах. **Базовая** использует исходные настроенные значения. **Агрессивная** понижает границу зоны сохранения, повышает буферный коэффициент и поддерживает высокий расход: вода как можно быстрее направляется вниз по течению.


In [ ]:
# @title Рисунок: влияние правил Каттакургана
#| label: fig-release-sensitivity
#| fig-cap: "Влияние правил Каттакурганского водохранилища на запасы (слева) и расход в HW_Karmana (справа). Консервативное правило сохраняет запасы, но ограничивает подачу в пик сезона. Агрессивное быстро опорожняет водохранилище, увеличивая пиковую подачу, но оставляя мало воды на осенний спад стока."
variants = kattakurgan_policy_variants(TIMESTEPS)
fig = plot_release_sensitivity(variants, TIMESTEPS)
figures.caption(
    fig,
    "Влияние правил Каттакурганского водохранилища на запасы (слева) и расход в HW_Karmana (справа). Консервативное правило сохраняет запасы, но ограничивает подачу в пик сезона. Агрессивное быстро опорожняет водохранилище, увеличивая пиковую подачу, но оставляя мало воды на осенний спад стока.",
    label="release-sensitivity",
)

Агрессивная стратегия быстро опорожняет водохранилище, увеличивая подачу вниз по течению в пик орошения, но оставляет мало запасов на спад стока в сентябре–октябре, когда нижние районы зависят от выпусков. Консервативная стратегия активно удерживает воду и сохраняет высокие запасы на всём протяжении расчёта, но ограничивает подачу именно тогда, когда оросительный спрос максимален. Базовый вариант находится между крайностями. **Единственного «правильного» правила водохранилища нет**: каждое выражает своё представление о том, когда вода наиболее ценна.


In [ ]:
# @title Таблица: правила Каттакургана и критерии системы
#| label: tbl-release-comparison
#| tbl-cap: "Влияние правил Каттакурганского водохранилища на критерии системы. Три правила дают существенно разные дефициты и сбросы, показывая компромисс между сохранением воды и нижним водоснабжением."
tables.caption(
    release_comparison_table(variants).style.hide(axis="index"),
    "Влияние правил Каттакурганского водохранилища на критерии системы. Три правила дают существенно разные дефициты и сбросы, показывая компромисс между сохранением воды и нижним водоснабжением.",
    label="release-comparison",
)

## Можно ли вручную улучшить справедливость распределения? {#sec-equity-manual}

В базовом варианте Карманаконимех — самый нижний район — обычно имеет наибольший дефицит. Можно ли исправить это, направляя больше воды вниз по течению?

HW_Narpay делит расход между двумя ветвями: к HW_Confluence (питающему HW_Karmana и далее Карманаконимех) и к району Нарпай. Базовая доля вычислена по спросу. Проверим два других распределения: с увеличением отвода к Нарпаю и с увеличением подачи вниз к Confluence.


In [ ]:
# @title Рисунок: районные дефициты при разных долях HW_Narpay
#| label: fig-equity-manual
#| fig-cap: "Районные дефициты при трёх долях HW_Narpay. Увеличение подачи к Confluence помогает нижнему Карманаконимеху; увеличение отвода к Нарпаю уменьшает местный дефицит, оставляя меньше нижнему коридору."
equity_variants = narpay_equity_variants(baseline, TIMESTEPS)
fig = plot_narpay_equity(equity_variants)
figures.caption(
    fig,
    "Районные дефициты при трёх долях HW_Narpay. Увеличение подачи к Confluence помогает нижнему Карманаконимеху; увеличение отвода к Нарпаю уменьшает местный дефицит, оставляя меньше нижнему коридору.",
    label="equity-manual",
)

Результаты показывают основной компромисс. Увеличение отвода к Нарпаю уменьшает местный дефицит, но может увеличить дефицит Карманаконимеха, поскольку меньше воды достигает HW_Karmana. И наоборот, увеличение подачи вниз помогает Карманаконимеху, но оставляет меньше Нарпаю. Базовая доля — один из возможных компромиссов, не единственный: последовательная структура сети означает, что **помощь одному району неизбежно ухудшает положение другого**. Многокритериальная оптимизация отвечает на эту основную проблему справедливости: вместо выбора одного компромисса мы показываем все достижимые варианты.

## Невозможный компромисс {#sec-impossible-tradeoff}

Можно ли найти *одно* изменение параметра, которое одновременно уменьшит общий дефицит (O1), сохранит нулевую нехватку воды для электростанции (O4) И улучшит справедливость распределения (O6)?

Ответ: нет. Каждая попытка улучшает один критерий за счёт другого. **Конфликты нескольких целей обусловлены структурой системы**, а не недостатком изобретательности. Именно для этого нужна оптимизация по Парето: она одновременно показывает *все* наилучшие достижимые компромиссы.

В @sec-optimisation-results оптимизатор исследует тысячи сочетаний параметров и покажет всю поверхность компромиссов. Но сначала точно определим, *чем* мы управляем (@sec-decision-space), *что* нам важно (@sec-objective-space) и *как* идёт поиск (@sec-optimisation-concepts).

::: {.callout-tip title="Упражнение 5: поиск компромиссов вручную" collapse="true"}
**Задание:** Выберите два параметра базового варианта, например `vr` Каттакургана и долю HW_Narpay. Создайте минимум 4 конфигурации, изменяя эти параметры. Для каждой выполните расчёт и запишите O1 (общий сельскохозяйственный дефицит) и O6 (справедливость / максимальный дефицит района). Нанесите не менее 4 попыток на диаграмму O1–O6.

1. Можно ли превзойти базовый вариант *одновременно по обоим* критериям?
2. Какие попытки «недоминируемы», то есть ни одна другая попытка не лучше по обоим критериям?
3. После @sec-pareto-front наложите полученный оптимизатором фронт Парето на свои попытки. Насколько близко вы подошли?

**Подсказка:** Создавайте независимые копии через `copy.deepcopy(system)`. Изменяйте правило выпуска через `object.__setattr__()`, как при настройке базового варианта.
:::

::: {.callout-important title="Главное — развитие интуиции"}
- Увеличение выпуска из Каттакургана помогает нижним районам, но грозит исчерпанием запасов до осеннего спада стока.
- Улучшение справедливости в одном узле увеличивает дефицит в другом: последовательная структура сети создаёт конфликты.
- Одно изменение параметра не может одновременно улучшить все критерии. Многокритериальная оптимизация позволяет показать эти компромиссы.
:::

# Пространство решений: чем мы можем управлять? {#sec-decision-space}

## Полный список настраиваемых величин {#sec-full-knobs}

В текущей задаче **124 настраиваемых скалярных значения**. Четыре — значения `eflow_fraction` распределителей; остальные 120 — месячно-циклические параметры SLOP двух водохранилищ (`vr`, `v1`, `v2`, `buffer_coef` и `flood_coef`, по 12 календарных месячных значений каждого параметра для каждого водохранилища). Расчёт остаётся суточным; месячные наборы задают компактное сезонное правило управления, повторяемое в суточном расчёте.


In [ ]:
# @title Таблица: перечень параметров оптимизации
#| label: tbl-full-knob-inventory
#| tbl-cap: "Текущий перечень параметров, извлечённый из WaterSystem.param_schema(). Параметры водохранилищ месячно-циклические, экологические доли распределителей скалярные."
schema_summary = parameter_schema_summary(baseline)
tables.caption(
    schema_summary.style.hide(axis="index"),
    "Текущий перечень параметров, извлечённый из WaterSystem.param_schema(). Параметры водохранилищ месячно-циклические, экологические доли распределителей скалярные.",
    label="full-knob-inventory",
)

Перечень параметров извлекается непосредственно из системы, а не поддерживается вручную в блокноте. Распределители **EFlowSplitPolicy** предоставляют один настраиваемый `eflow_fraction`, управляющий разделением между природным руслом и орошением; остаток распределяется по долям, взвешенным по спросу (рассчитанным по историческому спросу районов, а не оптимизированным). У распределителей **MonthlyDistribution** и **PriorityDistribution** распределение фиксировано. Каждое водохранилище добавляет пять семейств параметров SLOP по 12 сезонных значений. Ограничение `Ordered(v1 ≤ v2)` сохраняет правильный порядок границ зон каждого водохранилища.

На @fig-knob-map-full показана та же географическая сеть, что на @fig-network-map, но узлы Reach свёрнуты в направленные связи и выделены шесть настраиваемых компонентов. У каждого указано число скалярных значений, передаваемых оптимизатору.


In [ ]:
# @title Рисунок: шесть настраиваемых компонентов
#| label: fig-knob-map-full
#| fig-cap: "Географическая сеть с 6 настраиваемыми компонентами и числом их скалярных параметров; узлы Reach свёрнуты в направленные связи. Каждый распределитель даёт одну экологическую долю, водохранилища — месячно-циклические значения SLOP."
fig = plot_tunable_parameter_map(baseline, schema_summary)
figures.caption(
    fig,
    "Географическая сеть с 6 настраиваемыми компонентами и числом их скалярных параметров; узлы Reach свёрнуты в направленные связи. Каждый распределитель даёт одну экологическую долю, водохранилища — месячно-циклические значения SLOP.",
    label="knob-map-full",
)

## Выбор структуры: от настроек к параметрам {#sec-reducing-complexity}

Пространство решений — осознанный компромисс между реалистичностью и понятностью. Мы не оптимизируем произвольные суточные доли во всей сети, а выделяем небольшое число переменных правил распределения и сезонных параметров водохранилищ.

**EFlowSplitPolicy (4 распределителя коридора, по 1 параметру = 4 параметра).** Каждый использует `EFlowSplitPolicy`, разделяющее воду на два потока: доля `eflow_fraction` направляется на природные/экологические участки, остаток `(1 - eflow_fraction)` распределяется между оросительными участками по спросу. Веса рассчитаны по историческому спросу районов, а не оптимизируются. Вместо N−1 свободных долей каждого распределителя остаётся один скаляр, что особенно сокращает размерность у HW_Ravatkhoza (4 получателя) и HW_Damkhoza (4 получателя). Физическое обоснование: значимое управленческое решение — соотношение природы и орошения; распределение оросительной доли между районами лучше связывать со спросом, чем с неограниченной оптимизацией.

**PriorityDistribution в HW_Dargom (0 параметров).** Нормативный минимальный расход к Sink_Kashkadarya обеспечивается первым, остаток поступает в район Даргом. Это жёсткое эксплуатационное ограничение, не настраиваемый параметр.

**Фиксированные правила `MonthlyDistribution` в HW_Narpay и HW_Confluence (по 0 параметров).** В текущей суточной модели эти правила содержат 365-дневный цикл долей. Для базового варианта одна и та же взвешенная по спросу доля повторяется каждый день. HW_Narpay распределяет воду между 2 получателями с фиксированными долями. У HW_Confluence один получатель (HW_Karmana), поэтому выбора распределения нет.

**Месячно-циклическое SLOP обоих водохранилищ (по 60 значений = 120).** RES_Kattakurgan и RES_Akdarya используют `ZRBReleaseRule` с пятью сезонными семействами: `vr`, `v1`, `v2`, `buffer_coef`, `flood_coef`. Каждое содержит 12 календарных месячных значений, повторяющихся в суточном расчёте. Ограничение `Ordered(v1 ≤ v2)` удерживает верх буферной зоны ниже верха зоны сохранения запасов.

::: {.callout-note title="Почему мы не оптимизируем потери в каналах?" collapse="true"}
Около \~578 параметров потерь (коэффициенты фильтрации, размеры каналов, типы облицовки) характеризуют существующую инфраструктуру. Их изменение означает физическую реконструкцию каналов — инвестиционное, а не эксплуатационное решение. Оптимизатор управляет эксплуатацией: распределением и выпуском воды при существующей инфраструктуре.
:::


In [ ]:
# @title Таблица: переменные и границы
#| label: tbl-reduced-config
#| tbl-cap: "Переменные правил управления и объявленные границы."
tables.caption(
    decision_table_from_schema(schema_summary).style.hide(axis="index"),
    "Переменные правил управления и объявленные границы.",
    label="reduced-config",
)

Границы в @tbl-reduced-config соответствуют `__bounds__`, объявленным в каждом классе стратегии. **Доли экологического стока** лежат в [0, 1]: 0 означает отсутствие зарезервированной доли экологического коридора, 1 — резервирование всей воды для экологических направлений до заданного предела. **SLOP `vr`** меняется от 0 до 500 m³/s. **`v1` и `v2`** — от 0 до 10⁹ m³; `Ordered(v1 ≤ v2)` обеспечивается встроенной функцией исправления taqsim. **`buffer_coef`** находится в [0, 1], **`flood_coef`** — в [0, 5]. Оптимизатор выбирает значения в этих границах, а модель обеспечивает физические ограничения запасов, мёртвого объёма, ёмкости и потерь.

Важно различать **суточный шаг моделирования** и **месячно-циклическое правило управления**. Оптимизатор не задаёт 2191 суточное значение водохранилища. Он задаёт 12 сезонных значений каждого семейства SLOP, а правило выпуска применяет нужное значение на каждом суточном шаге.

::: {.callout-tip title="Упражнение D: подсчёт переменных решения" collapse="true"}
**Задание:** В текущей системе 4 доли экологического стока распределителей и 120 значений SLOP водохранилищ.

1. Проверьте число 120 по пяти семействам SLOP и 12 календарным месячным значениям каждого семейства.
2. Какой размер популяции для 124 переменных даёт практическое правило `pop_size ≥ 10 × variables`?
3. Что это означает для времени расчёта и охвата фронта Парето?
4. Как объединение месяцев в сезоны может уменьшить размерность при сохранении суточного расчёта?
:::

::: {.callout-tip title="Упражнение E: приоритеты и сигналы оптимизатора" collapse="true"}
**Задание:** HW_Karmana предоставляет параметр `eflow_fraction`, а не произвольные доли ветвей.

1. Объясните, как изменение `eflow_fraction` в HW_Karmana влияет на надёжность охлаждения Powerplant, хотя критерий электростанции рассчитывается ниже по течению.
2. Потери на связи HW_Karmana → Powerplant уменьшают фактически доставленный расход. Является ли O4 чисто сигналом верхней части сети или местная инфраструктура тоже важна? Объясните.
3. Если разрешить прямую оптимизацию всех долей ветвей HW_Karmana, какой новый компромисс появится? Стоит ли он увеличения размерности?
:::

::: {.callout-important title="Главное — пространство решений"}
- Текущая схема содержит 124 скалярных значения: 4 доли экологического стока и 120 месячно-циклических значений SLOP.
- EFlowSplitPolicy заменяет N-1 долей одним параметром экологического стока; остаток распределяется по спросу.
- Ограничения Ordered(v1 ≤ v2) обоих водохранилищ обеспечиваются встроенным исправлением taqsim.
- Переменные управляют правилами распределения и выпусков, без отдельной оптимизации каждого суточного значения.
:::

# Пространство критериев: что для нас важно? {#sec-objective-space}

## Набор возможных критериев {#sec-objective-landscape}

Для бассейна Зарафшана обсуждаются девять критериев, отражающих разные стороны работы системы. Текущая исполняемая оптимизация использует четыре критерия из `build_objectives()`: O1, O4, O6 и O9.


In [ ]:
# @title Таблица: девять возможных критериев
#| label: tbl-objective-landscape
#| tbl-cap: "Все девять возможных критериев бассейна Зарафшана. Четыре выбранных для оптимизации выделены."
tables.caption(
    objective_catalog_table().style.hide(axis="index"),
    "Все девять возможных критериев бассейна Зарафшана. Четыре выбранных для оптимизации выделены.",
    label="objective-landscape",
)

Каждый критерий описывает отдельную сторону работы системы. Прежде чем выбирать критерии оптимизации, необходимо понять, что каждый измеряет и почему это важно.

**O1 — Общий сельскохозяйственный дефицит** суммирует неудовлетворённый оросительный спрос всех шести районов за все 2191 суток. Функция `total_agricultural_deficit()` агрегирует события `DeficitRecorded` каждого узла спроса. Это основной показатель: на орошение приходится более 90% водопользования бассейна, и уменьшение дефицита непосредственно поддерживает производство продовольствия и сельские средства к существованию. O1 выбран для оптимизации.

**O2 — Дефицит экологического стока ниже по течению** измерял бы, насколько расход в Sink_Navoi ниже нормативного минимума для экологических и коммунальных нужд. Он отражает конфликт между верхними оросительными водозаборами и нижними обязательствами. В текущие четыре исполняемых критерия он не включён, поскольку в основном определяется верхними распределениями, уже отражёнными в O1 и O4; его включение увеличило бы размерность без независимой информации о компромиссах.

**O3 — Непроизводительные сбросы водохранилищ** суммировал бы неконтролируемые сбросы обоих водохранилищ. Это вода, которую невозможно сохранить или выпустить через регулируемые сооружения. Критерий связан с O1 (хорошее распределение означает меньше таких сбросов) и не выбран отдельно.

**O4 — Дефицит охлаждающей воды электростанции** измеряет накопленную нехватку в Powerplant относительно минимально необходимого расхода 15 m³/s. `passthrough_min_flow_deficit()` учитывает, как часто и на сколько воды не хватает. O4 отражает верхние распределения и местное правило HW_Karmana, определяющее остаток для электростанции после экологического и оросительного распределения. O4 выбран для оптимизации.

**O5 — Дефициты межбассейновых перебросок** измерял бы нехватку в трёх конечных узлах (Sink_Jizzakh, Sink_Kashkadarya, Sink_Navoi), представляющих обязательства перед соседними бассейнами и нормативные минимальные расходы. Как и O2, эти расходы в основном являются остатками верхних распределений и не добавляют независимых измерений компромисса. Не выбран.

**O6 — Справедливость (максимальный районный дефицит)** берёт максимум накопленных дефицитов шести районов. O1 измеряет сумму, O6 — положение самого неблагополучного района. При малом общем дефиците, сосредоточенном в одном районе, O1 будет хорошим, а O6 плохим. Функция `equity_deficit()` отражает справедливость, которую не показывают суммарные показатели. O6 выбран для оптимизации.

**O7 — Концентрация дефицита во времени** измерял бы, распределена ли нехватка равномерно в течение года или сосредоточена в нескольких критических месяцах. Надёжная большую часть года система с катастрофическим отказом в пик сезона получила бы худшую оценку, чем система с умеренным, но постоянным дефицитом. Критерий пока не реализован.

**O8 — Устойчивость запасов** измерял бы сопоставимость конечных и начальных запасов водохранилищ. Правило, достигающее малых дефицитов ценой полного опорожнения, неустойчиво на многолетнем горизонте. Критерий пока не реализован.

**O9 — Изменчивость выпуска** измеряет суммарные изменения суточных объёмов выпуска Каттакургана от одного дня к следующему. `release_variability()` штрафует нерегулярные выпуски. Реальные операторы не могут выполнять правила с резкими скачками между шагами: плавность работы — практическое ограничение. O9 выбран для оптимизации.

Все девять критериев сформулированы как задачи **минимизации**: дефицит, сброс, изменчивость, максимальный дефицит — везде меньше значит лучше. Это соглашение, не ограничение. Максимизацию можно превратить в минимизацию сменой знака: минимизация $-O$ математически эквивалентна максимизации $O$. NSGA-II и большинство систем многокритериальной оптимизации по умолчанию предполагают минимизацию. Единая формулировка упрощает реализацию и исключает смену знаков в функции оценки. При интерпретации помните: **меньшее значение всегда лучше** на каждой оси критериев.

## Выбор четырёх критериев {#sec-four-objectives}

Мы выбираем **O1, O4, O6, O9** по следующим причинам:

- **O1 (общий сельскохозяйственный дефицит)** отражает основную функцию системы: орошение составляет \>90% водопользования.
- **O4 (дефицит охлаждения электростанции)** — почти жёсткое ограничение. Он показывает, достаточно ли воды проходит через нижнюю сеть и местное правило HW_Karmana для охлаждения. Потери на связи HW_Karmana → Powerplant тоже влияют, поэтому это не исключительно верхний сигнал.
- **O6 (справедливость)** отражает аспект, отсутствующий в O1. Общий дефицит может быть малым, даже если пожертвовать одним районом.
- **O9 (изменчивость выпуска)** обеспечивает практическую выполнимость. Правило с резкими суточными скачками не может исполняться реальными операторами.

::: {.callout-note title="Почему не включить все девять критериев?" collapse="true"}
Больше критериев — выше размерность фронта, сложнее охватить его конечной популяцией и показать графически. O2 и O5 в основном определяются остатком после орошения и накопления: они увеличивают размерность без независимой информации. O3 связан с O1 (хорошее распределение даёт меньше сбросов). O7 и O8 пока не реализованы.
:::


In [ ]:
# @title Выбор четырёх критериев · код
#| label: objectives-setup
#| echo: true
OBJECTIVES = build_objectives(timesteps=TIMESTEPS)
OBJ_NAMES = [obj.name for obj in OBJECTIVES]
baseline_scores = compute_baseline(baseline, OBJECTIVES, timesteps=TIMESTEPS)
reference_point = compute_reference_point(baseline_scores)

## Значения критериев базового варианта {#sec-baseline-objectives}


In [ ]:
# @title Таблица: критерии базового варианта
#| label: tbl-baseline-objectives
#| tbl-cap: "Значения критериев базовой системы с постоянными настроенными долями — ориентиры для улучшения оптимизатором."
tables.caption(
    baseline_objective_table(baseline_scores, OBJ_NAMES).style.hide(axis="index"),
    "Значения критериев базовой системы с постоянными настроенными долями — ориентиры для улучшения оптимизатором.",
    label="baseline-objectives",
)

Базовый вариант даёт конкретный ориентир. Оптимизатор ищет настройки, улучшающие эти значения, учитывая, что улучшение одного критерия обычно ухудшает другой.

::: {.callout-tip title="Упражнение F: добавление пятого критерия" collapse="true"}
**Задание:** Предположим, O2 (дефицит экологического стока в Sink_Navoi) добавлен как пятый критерий.

1. Текущий фронт четырёх критериев — трёхмерная гиперповерхность. Какова размерность фронта пяти критериев? Какой минимальный `pop_size` вы рекомендуете по правилам из @sec-optimizer-config?
2. O2 измеряет нехватку расхода в Sink_Navoi. Электростанция (O4) тоже зависит от поступления воды в нижнюю сеть. Будут ли O2 и O4 связаны положительно, отрицательно или независимы? Как сильная связь повлияет на форму фронта?
3. Если O2 и O5 реализовать одинаковым способом через дефицит минимального расхода конечных узлов, чем они будут различаться? Можно ли объединить их без потери информации?
:::

::: {.callout-important title="Главное — пространство критериев"}
- Девять возможных критериев отражают разные стороны работы системы.
- Выбраны четыре (O1, O4, O6, O9): количество, надёжность, справедливость и управляемость.
- Базовый вариант задаёт ориентир для улучшения.
:::

# Понятия оптимизации {#sec-optimisation-concepts}

## Фронт Парето: формальное определение {#sec-pareto-formal}

После интуитивного знакомства в @sec-pareto-idea определим оптимальность по Парето формально.

**Доминирование.** Решение $\mathbf{a}$ *доминирует* решение $\mathbf{b}$ (обозначается $\mathbf{a} \prec \mathbf{b}$), если и только если:

- $\mathbf{a}$ не хуже $\mathbf{b}$ по каждому критерию **и**
- $\mathbf{a}$ строго лучше $\mathbf{b}$ хотя бы по одному критерию.

**Фронт Парето.** Фронт Парето (недоминируемое множество) состоит из решений, не доминируемых никаким другим допустимым решением. Недоминирующая сортировка присваивает каждому решению *ранг*: 0 — фронт Парето, 1 — фронт после удаления ранга 0 и так далее.

**Гиперобъём.** Индикатор гиперобъёма измеряет «объём» пространства критериев, доминируемый фронтом относительно опорной точки. Больший гиперобъём означает более сошедшийся и лучше распределённый фронт. Мы отслеживаем его по поколениям. Схематическое изображение дано на @fig-pareto-concept.

**Опорную точку** нужно выбирать внимательно. Кажется естественным взять значения базового варианта, но это не работает. Смысл многокритериальной оптимизации — *компромиссы*: решение с отличным O1 (малый сельскохозяйственный дефицит) может намеренно ухудшить O6 (справедливость) или O9 (плавность). Оно может быть *хуже* базового по отдельным критериям и намного лучше по другим, оставаясь недоминируемым: ни одно другое решение не превосходит его сразу по всем четырём критериям. Если опорная точка равна базовому значению на каждой оси, решение, превышающее его хотя бы по одному критерию, выходит за границу гиперобъёма, и его вклад обрезается или теряется. Качество фронта будет недооценено.

Поэтому опорная точка каждого критерия равна **1.5× его базового значения**. Множитель оставляет запас для всего фронта, включая решения, жертвующие одним критерием ради других. Опорная точка — «заведомо плохой» верхний предел, который любое разумное распределение должно превосходить по каждой оси, а не целевой показатель. Масштабы критериев сильно различаются: O1, O4 и O6 — накопленные объёмы дефицита, O9 — сумма суточных изменений выпуска. Множитель 1.5× применяется отдельно к каждому критерию для пропорционального запаса.

**Точки перегиба** — решения, где наклон компромисса меняется наиболее резко: наибольший результат на единицу уступки. Удаление от такой точки в любом направлении даёт малое улучшение одного критерия ценой большого ухудшения другого.


In [ ]:
# @title Рисунок: формальное представление фронта Парето
#| label: fig-pareto-concept
#| fig-cap: "Двумерный фронт Парето: доминирование, недоминируемый фронт (ранг 0), точка перегиба и гиперобъём (затенённая область)."
fig = plot_pareto_concept()
figures.caption(
    fig,
    "Двумерный фронт Парето: доминирование, недоминируемый фронт (ранг 0), точка перегиба и гиперобъём (затенённая область).",
    label="pareto-concept",
)

## NSGA-II {#sec-nsga2}

Поиск оптимальных по Парето решений требует алгоритма, исследующего многомерное пространство и одновременно оптимизирующего четыре конфликтующие цели. Мы используем **генетический алгоритм**, вдохновлённый биологической эволюцией. Начинаем с *популяции* кандидатов (разных векторов параметров), оцениваем каждый по четырём критериям, преимущественно *отбираем* лучших, *скрещиваем* их, получая потомков с признаками обоих родителей, и слегка *мутируем* значения для исследования новых областей. Через многие поколения популяция приближается к фронту Парето.

Для Зарафшана каждая «особь» — вектор долей экологического стока и месячно-циклических значений SLOP. Скрещивание может объединить правило водохранилища одного родителя и экологические доли другого. Мутация вносит малые изменения, например +2 m³/s к выпуску или +10 Mm³ к границе зоны, не позволяя популяции застрять в локальном оптимуме.

В многокритериальной задаче нет единственного «лучшего» решения. **NSGA-II** (Non-dominated Sorting Genetic Algorithm II) [@deb2002nsga2] заменяет простой рейтинг качества двумя механизмами. **Недоминирующая сортировка** распределяет особи по слоям Парето: ранг 0 — текущие недоминируемые решения, ранг 1 — те, которые стали бы недоминируемыми после удаления ранга 0, и так далее. При отборе родителей предпочтительнее меньший ранг. **Расстояние скученности** показывает изолированность решения внутри ранга: редкие области пространства предпочтительнее плотных скоплений. Это одновременно приближает популяцию к истинному фронту (сходимость) и распределяет её вдоль него (разнообразие), создавая хорошее приближение всей поверхности компромиссов. NSGA-II стал стандартным алгоритмом многокритериальной оптимизации водных ресурсов [@maier2014evolutionary; @reed2013evolutionary].

Мы используем `taqsim.optimization.optimize()` через обёртку пакета `run_optimization()`. Применяются скрещивание SBX, полиномиальная мутация и автоматически созданная функция исправления. Оптимизатор принимает критерии, число шагов, размер популяции, число поколений, зерно случайных чисел и число параллельных исполнителей; он управляет популяцией, отбором, отслеживанием поколений и записью фронта.

## Вопрос оптимизации {#sec-opt-question}

::: {.callout-important title="Вопрос оптимизации"}
*Какая комбинация долей экологического стока и месячно-циклических параметров SLOP минимизирует сельскохозяйственный дефицит (O1), сохраняя надёжность электростанции (O4), справедливость между районами (O6) и достаточно плавную работу для практического исполнения (O9)?*

*Единственного ответа нет. Фронт Парето показывает все наилучшие достижимые компромиссы.*
:::

::: {.callout-tip title="Упражнение G: влияние числа настроек" collapse="true"}
**Задание:** Сейчас используются месячно-циклические значения SLOP. Предположим, вместо 12 месяцев заданы четыре сезонных значения каждого семейства.

1. Сколько значений SLOP останется?
2. Сместится ли фронт ближе к началу координат, дальше или останется тем же? Почему?
3. Какое практическое преимущество есть у четырёх сезонов, даже если 12 месяцев дают лучшие оценки?
:::

::: {.callout-important title="Главное — понятия оптимизации"}
- На фронте Парето нельзя улучшить один критерий, не ухудшив другой.
- Гиперобъём оценивает сходимость и распределение фронта.
- NSGA-II использует недоминирующую сортировку и расстояние скученности для разнообразного приближения фронта.
- Функция исправления обеспечивает `Ordered(v1 ≤ v2)` после каждой генетической операции.
:::

# Результаты оптимизации {#sec-optimisation-results}

## Настройка оптимизатора {#sec-optimizer-config}

Каждый кандидат — вектор параметров правил управления. Оптимизатор предлагает вектор, система получает эти значения, выполняет 2191 суточный шаг и оценивается по 4 критериям. Лучшие решения скрещиваются и мутируют, создавая следующее поколение.

::: {.callout-note title="Как taqsim находит параметры" collapse="true"}
Метод `WaterSystem.param_schema()` автоматически находит настраиваемые параметры, обходя дерево узлов и собирая поля, объявленные в `__params__`. Для нашей системы получается схема из @tbl-full-knob-inventory: `eflow_fraction` распределителей и месячно-циклические значения SLOP.

`WaterSystem.constraint_specs()` находит ограничения между параметрами: здесь 2 ограничения `Ordered(v1 ≤ v2)`. Фабрика `make_repair()` создаёт функцию, обеспечивающую их после каждой генетической операции: если `v1 > v2`, значения меняются местами.

`with_vector()` создаёт неизменяемую копию системы с новым вектором параметров, исключая ручное сочетание `deepcopy` и `object.__setattr__()`.
:::


In [ ]:
# @title Настройка оптимизатора · код
#| label: optimizer-problem-setup
#| echo: true
#| eval: false
system = create_zrb_system(use_canal_losses=True)
apply_static_demand_splitting(system, verbose=False)
system.validate()

# Inspect what the optimizer will tune
schema = system.param_schema()
print(f"Tunable parameters: {len(schema)}")
for spec in schema:
    print(f"  {spec.path} = {spec.value:.4f}")

# Inspect constraints
constraints = system.constraint_specs()
print(f"\nConstraints: {len(constraints)}")
for spec in constraints:
    print(f"  {type(spec.constraint).__name__}({spec.constraint.params}) @ {spec.prefix}")

In [ ]:
# @title Настройка оптимизатора · код
#| label: run-optimization
#| echo: true
#| eval: false
FORCE_RERUN = False
SMOKE_TEST = True

result, convergence_history, config = run_coursebook_optimization(
    system,
    OBJECTIVES,
    OBJ_NAMES,
    reference_point,
    timesteps=TIMESTEPS,
    seed=SEED,
    smoke_test=SMOKE_TEST,
    force_rerun=FORCE_RERUN,
)
print(f"Pareto front: {len(result.solutions)} non-dominated solutions")

In [ ]:
# @title Извлечение массивов критериев из результата
#| label: extract-front
#| echo: true
#| eval: false
# Extract objective arrays from the result.
front_obj, n_front = front_arrays(result, OBJ_NAMES)
print(f"Pareto front: {n_front} non-dominated solutions")

Размер популяции должен поддерживать разнообразие всего фронта. Для 4 критериев фронт — трёхмерная гиперповерхность, которую гораздо сложнее охватить, чем одномерную кривую. При выборе размера используется следующее практическое правило.

| Критерии | Геометрия фронта | Типичный pop_size |
|:-----------|:----------------|:-----------------|
| 2 | Одномерная кривая | 50--100 |
| 3 | Двумерная поверхность | 100--200 |
| 4 | Трёхмерная гиперповерхность | 150--300 |

Второе практическое правило требует популяции хотя бы в 10 раз больше числа переменных решения. Для текущей схемы такой запуск слишком велик для учебного блокнота в реальном времени. Поэтому по умолчанию `SMOKE_TEST = True`: крошечный запуск `12 × 2` проверяет только работоспособность всей последовательности. Для полноценной учебной демонстрации задайте `SMOKE_TEST = False`: используются `POP_SIZE = 300` и `N_GENERATIONS = 100`. Для фронтов публикационного качества нужны повторные запуски с большей популяцией и большим числом поколений.

Число поколений определяет время на уточнение и сходимость фронта. Проверочный режим выполняет лишь 24 оценки. Полная демонстрация: 300 особей × 100 поколений = 30,000 оценок.

## Проверка сходимости {#sec-convergence}

Перед изучением фронта нужно проверить сходимость: дополнительные поколения не должны существенно улучшать его. Основной показатель — **гиперобъём**, объём пространства критериев, доминируемый текущим фронтом относительно опорной точки из @sec-pareto-formal. По мере работы фронт улучшается, гиперобъём растёт. Когда кривая выходит на плато и перестаёт расти от поколения к поколению, фронт стабилизировался и дальнейшие расчёты дают всё меньшую отдачу.

На @fig-convergence две проверки. Слева — гиперобъём по поколениям: быстрый рост в начале (быстро находятся хорошие решения), затем плато (сходимость). Если кривая продолжает расти на поколении 100, следует увеличить `N_GENERATIONS` до 200 или 300 и повторить расчёт. Справа — число недоминируемых решений, то есть размер фронта, по поколениям. Обычно оно быстро растёт при заполнении фронта и стабилизируется, когда новые решения заменяют существующие, а не расширяют фронт.


In [ ]:
# @title Рисунок: сходимость гиперобъёма
#| label: fig-convergence
#| eval: false
#| fig-cap: "Сходимость гиперобъёма по поколениям. Выход на плато означает стабилизацию фронта Парето."
if convergence_history.records:
    fig = plot_convergence(convergence_history)
    figures.caption(
        fig,
        "Сходимость гиперобъёма по поколениям. Выход на плато означает стабилизацию фронта Парето.",
        label="convergence",
    )
else:
    print("No convergence history available; cached results may have been loaded without callback records.")

Если гиперобъём не вышел на плато, увеличьте `N_GENERATIONS`, например до 200 или 300, и задайте `FORCE_RERUN = True`. Каждое поколение добавляет `POP_SIZE` оценок, поэтому время почти линейно растёт и с размером популяции, и с числом поколений.

## Фронт Парето {#sec-pareto-front}

На @fig-pareto шесть диаграмм — по одной для каждой пары из четырёх критериев. Каждая синяя точка — решение фронта: ни одно другое решение популяции не лучше сразу по *всем четырём* критериям. Красная звезда — базовый вариант с постоянными настроенными долями.


In [ ]:
# @title Рисунок: попарные диаграммы четырёх критериев
#| label: fig-pareto
#| eval: false
#| fig-cap: "Попарные диаграммы фронта Парето четырёх критериев. Синяя точка — недоминируемое решение, красная звезда — базовый вариант с постоянными настроенными долями. Шесть панелей показывают все C(4,2) = 6 пар критериев."
fig = plot_pareto_scatter(result, OBJ_NAMES, baseline_scores=baseline_scores)
figures.caption(
    fig,
    "Попарные диаграммы фронта Парето четырёх критериев. Синяя точка — недоминируемое решение, красная звезда — базовый вариант с постоянными настроенными долями. Шесть панелей показывают все C(4,2) = 6 пар критериев.",
    label="pareto",
)

### Как читать матрицу диаграмм рассеяния {#sec-reading-scatter-matrix}

Первый вопрос часто таков: «Почему не соединить точки линией фронта Парето?». Для двух критериев это правильно. Фронт — одномерная кривая; сортировка по одному критерию упорядочивает точки вдоль фронта. Соединив их слева направо, получим знакомую вогнутую кривую компромисса из учебников.

С четырьмя критериями всё иначе. Фронт лежит в **четырёхмерном пространстве критериев** и образует **трёхмерную гиперповерхность** (на одно измерение меньше числа критериев). Каждая панель проецирует её на плоскость, отбрасывая два критерия. Это имеет три важных следствия:

**1. Внутренние точки не хуже остальных.** Решение «внутри» облака O1–O4 не доминируется: оно принадлежит истинному четырёхмерному фронту. Оно выглядит внутренним, поскольку превосходно по O6 или O9, *не показанным* на панели. Двумерная выпуклая оболочка только граничных точек O1–O4 визуально исключила бы действительно оптимальные решения. Они выглядели бы неудачами, хотя представляют осознанные компромиссы по скрытым критериям.

**2. Доминирование нельзя переносить между панелями.** Граничная точка O1–O4 может быть глубоко внутри облака O6–O9, и наоборот. Ни одна двумерная оболочка не показывает все компромиссы. Чтобы понять, *почему* решение на фронте, нужно рассматривать все шесть панелей вместе или параллельные координаты в @sec-parallel-coordinates, где видны все критерии.

**3. Форма облака раскрывает структуру компромисса.** Даже без соединяющих линий панели информативны:

- **Тесная отрицательная связь** (диагональ от верхнего левого к нижнему правому углу): сильный прямой конфликт. Улучшение одного критерия устойчиво ухудшает другой. Ищите это для O1–O4: уменьшение сельскохозяйственного дефицита увеличивает нехватку охлаждающей воды.
- **Широкое облако без явного направления**: критерии относительно независимы или их связь определяется скрытыми критериями. Возможен широкий диапазон обоих значений.
- **Треугольное или Г-образное облако**: один критерий достиг жёсткого физического предела, другой свободно меняется. Так проявляются ограничения структуры системы.
- **Положение базового варианта**: красная звезда относительно облака показывает улучшение по сравнению с простым правилом. Если звезда внутри или у края, исходное правило уже близко к оптимальному по этой паре. Если далеко, выигрыш существенен.

::: {.callout-note title="Почему шесть панелей?" collapse="true"}
Для $k$ критериев имеется $\binom{k}{2}$ уникальных пар; для 4 получается $\binom{4}{2} = 6$ панелей. Это минимум двумерных видов для всех попарных связей. Однако даже все 6 не полностью представляют трёхмерную гиперповерхность, как шесть фотографий скульптуры не показывают все ракурсы. Параллельные координаты (@fig-parallel-coordinates) дополняют матрицу, показывая четыре критерия в одной системе осей и позволяя проследить компромисс каждого решения.
:::

Упражнение 6 (@sec-pareto-front) предлагает применить эти приёмы к матрице выше.

## Четырёхмерный фронт Парето в трёхмерном представлении {#sec-3d-pareto}

Матрица проецирует четырёхмерный фронт на шесть плоскостей. Дополнительный вид показывает три критерия одновременно во вращаемой трёхмерной диаграмме, а четвёртый — цветом.

На @fig-pareto-3d ось x — O1 (сельскохозяйственный дефицит), y — O4 (дефицит электростанции), z — O6 (справедливость). Цвет точки задаётся O9 (плавность). Цвет служит четвёртым визуальным измерением.

Вращайте вид нажатием и перетаскиванием. Обратите внимание: точки, выглядевшие «внутренними» на плоских панелях, лежат на поверхности трёхмерного облака — они недоминируемы, но скрытые измерения смещают их проекцию внутрь. Посмотрите, сосредоточены ли определённые цвета в одной области: требование более плавной работы (низкий O9) может ограничивать компромиссы остальных трёх критериев.


In [ ]:
# @title Рисунок: интерактивный трёхмерный фронт
#| label: fig-pareto-3d
#| eval: false
#| fig-cap: "Интерактивный трёхмерный фронт: O1 × O4 × O6, цвет — O9. Перетаскивайте для вращения."
fig = plot_pareto_3d(result, OBJ_NAMES)
figures.caption(
    fig,
    "Интерактивный трёхмерный фронт: O1 × O4 × O6, цвет — O9. Перетаскивайте для вращения.",
    label="pareto-3d",
)

## Интерактивные параллельные координаты {#sec-parallel-coordinates}

Параллельные координаты — наиболее мощный инструмент исследования многокритериального фронта. На @fig-parallel-coordinates каждая вертикальная ось — один критерий или важная переменная решения. Ломаная, соединяющая значения на осях, — одно решение фронта. Цвет кодирует O1 (общий сельскохозяйственный дефицит): тёмные линии соответствуют малому, светлые — большому дефициту.

**Как взаимодействовать.** Нажмите и протяните вдоль вертикальной оси, выделяя диапазон. Останутся только решения с соответствующим значением. Можно сочетать фильтры: сначала выделите нулевой дефицит электростанции на O4, затем ограничьте O6 для справедливого распределения. Оставшиеся линии покажут достижимые O1 и O9 и соответствующие параметры при совместных ограничениях. Чтобы снять фильтр, нажмите на ось вне выделенного диапазона.

**Что можно узнать.** Примерно параллельные, не пересекающиеся линии между осями указывают на положительную связь: улучшение одного критерия обычно улучшает другой. Много пересечений означает конфликт и широкий спектр компромиссов. Если на оси переменной все линии близки, значение «зафиксировано» физикой: оптимизатор выбирает его независимо от приоритетной цели. Широко распределённые значения — настоящие рычаги управления: разные стратегии требуют принципиально разных настроек.

**Для переговоров заинтересованных сторон** график показывает измеримую «цену» сочетания ограничений. Например: (1) представитель электростанции оставляет O4 с нулевым дефицитом; (2) представитель сельского хозяйства ограничивает O1 приемлемым порогом; (3) комиссия по справедливости ограничивает O6, чтобы ни один район не нёс чрезмерную долю нехватки. Оставшиеся линии показывают диапазон O9 и конкретные параметры распределения и водохранилищ. Так абстрактный анализ Парето становится инструментом переговоров: стороны сразу видят выигрыш и цену каждого дополнительного ограничения.


In [ ]:
# @title Рисунок: интерактивные параллельные координаты
#| label: fig-parallel-coordinates
#| eval: false
#| fig-cap: "Интерактивные параллельные координаты. Каждая линия — решение фронта Парето. Выделяйте диапазоны на осях для фильтрации. Цвет — O1, общий сельскохозяйственный дефицит."
decision_vars = list(result.solutions[0].parameters)[:8]
fig = plot_parallel_coordinates(result, OBJ_NAMES, decision_vars=decision_vars)
figures.caption(
    fig,
    "Интерактивные параллельные координаты. Каждая линия — решение фронта Парето. Выделяйте диапазоны на осях для фильтрации. Цвет — O1, общий сельскохозяйственный дефицит.",
    label="parallel-coordinates",
)

## Характерные решения {#sec-representatives}

Для практического выбора из множества точек полезно выделить несколько **характерных решений**, показывающих диапазон компромиссов. Мы выбираем пять: по одному минимуму каждого из четырёх критериев («крайние» решения) и **сбалансированный** вариант, ближайший к идеальной точке — гипотетическому одновременному минимуму всех критериев.

В @tbl-representatives приведены их оценки. Крайние показывают «цену» специализации: минимум сельскохозяйственного дефицита (Min Ag. Deficit) даёт наименьший O1, но может ухудшать O6 или O9. Min Equity даёт самое справедливое распределение ценой большего общего дефицита. Balanced — промежуточный вариант: не лучший ни по одному критерию, но приемлемый по всем четырём.


In [ ]:
# @title Таблица: характерные решения фронта
#| label: tbl-representatives
#| eval: false
#| tbl-cap: "Характерные решения фронта: минимумы каждого критерия по отдельности и сбалансированный компромисс."
representatives = representative_indices(front_obj, OBJ_NAMES)
tables.caption(
    representative_table(front_obj, representatives, OBJ_NAMES).style.hide(axis="index"),
    "Характерные решения фронта: минимумы каждого критерия по отдельности и сбалансированный компромисс.",
    label="representatives",
)

На @fig-district-breakdown O1 (общий сельскохозяйственный дефицит) разложен по районам для каждого характерного решения. Видно, какие районы платят за разные стратегии: суммарные оценки таблицы скрывают географическое распределение выгод и потерь.


In [ ]:
# @title Рисунок: районные дефициты характерных решений
#| label: fig-district-breakdown
#| eval: false
#| fig-cap: "Разложение дефицита по районам для характерных решений. Разные решения жертвуют разными районами, показывая географическую сторону компромисса."
rep_systems = simulate_representatives(result, representatives, TIMESTEPS)
fig = plot_representative_district_deficits(rep_systems)
figures.caption(
    fig,
    "Разложение дефицита по районам для характерных решений. Разные решения жертвуют разными районами, показывая географическую сторону компромисса.",
    label="district-breakdown",
)

Помимо орошения, решения влияют на важные для сторон **расходы, не являющиеся отдельными критериями**: межбассейновые переброски (конечные узлы) и доступность охлаждающей воды. На @fig-sink-powerplant показано суммарное поступление в каждый конечный узел и Powerplant для пяти решений. Эти расходы непосредственно не оптимизируются, а возникают как следствие распределений.


In [ ]:
# @title Рисунок: поступление в конечные узлы и Powerplant
#| label: fig-sink-powerplant
#| eval: false
#| fig-cap: "Суммарное поступление в конечные узлы и Powerplant для характерных решений. Эти расходы не оптимизируются напрямую, а следуют из распределений. Различия показывают, как верхние компромиссы влияют на межбассейновые обязательства и надёжность выработки электроэнергии."
fig = plot_representative_sink_flows(rep_systems, TIMESTEPS)
figures.caption(
    fig,
    "Суммарное поступление в конечные узлы и Powerplant для характерных решений. Эти расходы не оптимизируются напрямую, а следуют из распределений. Различия показывают, как верхние компромиссы влияют на межбассейновые обязательства и надёжность выработки электроэнергии.",
    label="sink-powerplant",
)

Расходы конечных узлов представляют межбассейновые переброски и нормативные минимумы: Sink_Jizzakh, Sink_Kashkadarya и Sink_Navoi подают воду соседним бассейнам, частично зависящим от Зарафшана. Расход Powerplant — доступность охлаждающей воды для выработки электроэнергии. Различия решений показывают, как верхние компромиссы по оптимизируемым критериям передаются нижним обязательствам, которые мы отслеживаем, но не оптимизируем напрямую.

## Чему «научился» оптимизатор? {#sec-optimizer-learned}

@fig-parameters показывает, что оптимизатор «узнал» о физической структуре системы. Каждая позиция x — выбранная переменная управления. Значение каждого характерного решения показано точкой и нормировано в [0, 1], где 0 — нижняя граница, 1 — верхняя. Это позволяет сравнивать переменные с разными единицами.

Главное — вертикальный разброс точек. Если все пять почти на одной высоте, это **параметры согласия**: оптимизатор получает практически одно значение независимо от приоритетной цели. Физика оставляет мало свободы: значение определяется структурой сети, пропускной способностью или распределением спроса, а не предпочтениями. Такие параметры можно применять уверенно как устойчивый эксплуатационный выбор.

Широкий разброс по [0, 1], напротив, показывает **настоящие рычаги управления**: разные компромиссы требуют разных значений. Именно здесь предпочтения имеют значение: приоритет справедливости даёт иные настройки, чем минимум общего дефицита. Различение параметров согласия и рычагов — возможно, самый практически полезный результат оптимизации: оно показывает управляющему, где есть реальная гибкость, а где решение уже определила физика.


In [ ]:
# @title Рисунок: переменные характерных решений
#| label: fig-parameters
#| eval: false
#| fig-cap: "Переменные характерных решений, нормированные в [0, 1]. Тесные вертикальные группы — параметры согласия, определяемые физикой; широкий разброс — реальные рычаги управления, зависящие от предпочтений сторон."
fig = plot_selected_parameter_profiles(result, representatives, system.param_bounds())
figures.caption(
    fig,
    "Переменные характерных решений, нормированные в [0, 1]. Тесные вертикальные группы — параметры согласия, определяемые физикой; широкий разброс — реальные рычаги управления, зависящие от предпочтений сторон.",
    label="parameters",
)

При интерпретации рассмотрите следующие вопросы:

- **Какая доля экологического стока сильнее всего отличается от выбора оператора?** Какое физическое условие — например спрос или нижняя структура сети — она компенсирует?
- **Предпочитает ли оптимизатор агрессивный или консервативный выпуск Каттакургана?** Как различается `buffer_coef` у характерных решений и что это говорит о ценности страховки от засухи?
- **По-разному ли используют Каттакурган решения с минимумом несправедливости и общего дефицита?** Что это говорит о роли запасов в справедливости?
- **Сравните режим выпуска самого плавного решения и решения с минимумом дефицита.** Какую эксплуатационную реальность обеспечивает критерий плавности?
- **Нашёл ли оптимизатор что-то неочевидное** — настройку, которую оператор мог не рассмотреть?

::: {.callout-tip title="Упражнение 6: чтение фронта Парето" collapse="true"}
**Задание:** Используя попарные диаграммы @fig-pareto:

1. Найдите пару с самым резким компромиссом: малое улучшение одного сильно ухудшает другой. Какой физический механизм вызывает конфликт?
2. Найдите положительно связанные критерии: улучшение одного улучшает другой. Не является ли один из них избыточным?
3. Сколько недоминируемых решений имеют O4 = 0? Каков их диапазон O1?
4. Выберите граничную точку O1–O4 и найдите её на O6–O9 (подсказка: параллельные координаты). Осталась ли она на границе или стала внутренней? Что это говорит о стратегии компромисса?
5. Для двух критериев, например O1 и O4, было бы правильно соединить точки непрерывной линией? В @sec-reading-scatter-matrix объяснено различие между 2 и 4 критериями.
:::

::: {.callout-tip title="Упражнение 7: работа с параллельными координатами" collapse="true"}
**Задание:** Используйте интерактивный график @fig-parallel-coordinates:

1. Выделите на O4 только нулевой дефицит электростанции. Сколько решений осталось? Каков диапазон O1?
2. Дополнительно оставьте O6 ниже медианы. Какова цена в O9 — какой диапазон плавности остался?
3. Рассмотрите оси переменных этого дважды отфильтрованного множества. Какая меняется меньше всего и сильнее «зафиксирована» ограничениями? Какая меняется больше и даёт наибольшую гибкость?
4. Подведите итог: какова «цена» совместного обеспечения надёжности электростанции и справедливости, выраженная через плавность и общий дефицит?
:::

::: {.callout-important title="Главное — результаты оптимизации"}
- Фронт Парето показывает все достижимые компромиссы четырёх критериев.
- Характерные решения объясняют, *как* настройки приводят к разным компромиссам.
- Интерактивные параллельные координаты — основной инструмент исследования фронта.
- Понимание того, *чему научился оптимизатор*, так же важно, как полученные оценки.
:::

# Обсуждение {#sec-discussion}

## Что показывает оптимизация {#sec-what-reveals}

Фронт количественно описывает основные компромиссы Зарафшана. Вернитесь к @sec-building-intuition: обнаруженные вручную компромиссы теперь систематически картированы. Оптимизатор не создал новую физику, а исследовал последствия, с которыми студенты уже познакомились.

Основные выводы о структуре:

- **Орошение и справедливость**: O1 и O6 конфликтуют, поскольку последовательная сеть систематически ухудшает положение нижних районов. Уменьшение суммы дефицитов концентрирует воду выше, где спрос обслуживается первым.
- **Надёжность и результативность**: нулевой O4 ограничивает распределение и может увеличить сельскохозяйственный дефицит. Местное правило HW_Karmana и нижние потери определяют, какая часть направленной воды фактически обслужит электростанцию.
- **Плавность и оптимальность**: O9 штрафует малые дефициты, достигнутые нерегулярной работой водохранилищ. Это эксплуатационно необходимо, но имеет цену.

## Выбор решения: дилемма министра водных ресурсов {#sec-ministers-dilemma}

> *Министерство должно выбрать одно правило с фронта Парето. За столом четыре стороны:*
>
> - *Отдел орошения хочет минимизировать общий сельскохозяйственный дефицит (O1).*
> - *Комиссия по справедливости требует, чтобы ни один район не нёс более 35% общего дефицита (ограничение O6).*
> - *Оператор электростанции требует нулевой нехватки охлаждающей воды (O4 = 0).*
> - *Эксплуатационный отдел требует суточных изменений выпуска ниже установленного им порога (ограничение O9).*
>
> *С помощью @fig-parallel-coordinates найдите множество решений, удовлетворяющих всем четырём сторонам. Сколько их? Каков достижимый диапазон O1? Если комиссия ослабит предел до 40%, сколько решений добавится и насколько улучшится лучший O1?*
>
> *Это иллюстрирует основной смысл многокритериальной оптимизации: фронт предоставляет меню, а стороны договариваются о выборе блюда.*

## Чего оптимизация не определяет {#sec-limitations-intro}

Фронт отвечает «что достижимо?», но не «что следует выбрать?». Выбор требует человеческих ценностей, переговоров и политического суждения. Оптимизатор поддерживает принятие решения, но не принимает его.

## Ограничения модели {#sec-model-limitations}

Следует помнить об упрощениях:

- **Полное знание будущего**: оптимизатор заранее знает весь 2191-дневный приток; реальные операторы сталкиваются с неопределённостью прогноза.
- **Один исторический ряд**: результат обусловлен гидрологией 2017–2022 годов. Другой климатический ряд даст другой фронт.
- **Суточный шаг**: внутрисуточная динамика — движение паводка и пики спроса — усредняется.
- **Непрерывные переменные**: реальные решения дискретны, например положения затворов и расписания выпусков.
- **Нет эксплуатационных затрат**: перемещение воды считается бесплатным.

## Дальнейшие направления {#sec-future-directions}

- **Другие сезонные параметризации**: замена 12 месячных значений SLOP четырьмя сезонами улучшит понятность и уменьшит размерность; сезонные экологические доли добавят гибкость и размерность.
- **Климатические сценарии**: искусственные засухи позволяют проверить устойчивость.
- **Робастная оптимизация**: одновременная оптимизация нескольких рядов притока выявит хорошие решения при неопределённости.
- **Группировка сезонов**: четыре блока SLOP вместо 12 месяцев могут согласовать гибкость и вычислительную выполнимость.

::: {.callout-tip title="Упражнение H: позиции заинтересованных сторон" collapse="true"}
**Задание:** Вы консультируете Министерство водных ресурсов Узбекистана по распределению Зарафшана. Ожидается засуха: приток источника уменьшится на 20%.

1. Какому из O1, O4, O6, O9 вы отдадите приоритет и почему? Изменится ли ответ при засухе длительностью 3 года?
2. Сейчас экологические доли скалярны, правила водохранилищ месячно-циклические. Какое преимущество и недостаток дадут сезонные экологические доли при засухе?
3. Нижняя область считает, что отсутствие O5 (дефицита перебросок) несправедливо отдаёт преимущество верхним районам. Что вы ответите?
:::

::: {.callout-important title="Главное — обсуждение"}
- Фронт количественно описывает компромиссы; выбор определяется человеческими ценностями.
- Оптимизация обусловлена допущениями модели и одним историческим рядом.
- В дальнейшем можно расширить решения, добавить климатические сценарии и проверить устойчивость.
:::

# Выводы {#sec-conclusions}

## Основные результаты {#sec-summary}

Пособие показало многокритериальную оптимизацию распределения воды в бассейне Зарафшана:

1. **Осознанный выбор параметров правил**: EFlowSplitPolicy заменяет доли распределителей одной экологической долей, а SLOP предоставляет сезонные значения вместо суточных управлений.
2. **Четыре критерия отражают главные компромиссы**: сельскохозяйственный дефицит (O1), надёжность электростанции (O4), справедливость (O6) и плавность (O9).
3. **Фронт выявляет структурные компромиссы**: справедливость конфликтует с общим дефицитом из-за последовательной сети. Плавность конфликтует с результативностью, поскольку оптимальная работа требует нерегулярных выпусков. Это физические реалии, а не артефакты модели.
4. **Параметры оптимизатора интерпретируемы**: исследуя настройки разных решений, можно понять, что он «узнал» и какие эксплуатационные изменения наиболее значимы.

## Практическое значение {#sec-practical-implications}

Для управляющего водными ресурсами:

- Фронт — меню, не предписание. Он показывает достижимое и его цену.
- Параллельные координаты — основной инструмент поддержки решений. Сначала фильтруйте жёсткие ограничения (O4 = 0, O9 ниже порога), затем исследуйте остаток.
- Мало меняющиеся параметры «зафиксированы» физикой: их настройка мало даст. Широко меняющиеся — настоящие рычаги.
- Приоритет HW_Dargom показывает, что некоторые решения лучше задавать однозначными правилами, чем оптимизацией. EFlowSplitPolicy показывает, как физические знания в структуре правил уменьшают размерность.

# Сборник упражнений {#sec-exercises}

## Концептуальные упражнения и обсуждение (буквы)

- **Упражнение D — подсчёт переменных решения** (@sec-reducing-complexity): размерность месячно-циклической схемы SLOP и возможное сезонное сокращение.
- **Упражнение E — приоритеты и сигналы оптимизатора** (@sec-reducing-complexity): как приоритет HW_Dargom и EFlowSplitPolicy уменьшают пространство оптимизации.
- **Упражнение F — пятый критерий** (@sec-four-objectives): влияние O2 на размерность фронта и связи критериев.
- **Упражнение G — число настроек** (@sec-opt-question): последствия замены месячных значений SLOP крупными сезонными группами.
- **Упражнение H — позиции сторон** (@sec-future-directions): обсуждение приоритетов при засухе.

## Расчётные упражнения (номера)

5. **Упражнение 5 — ручной поиск компромиссов** (@sec-impossible-tradeoff): измените два параметра, нанесите попытки в пространство критериев и сравните с фронтом.
6. **Упражнение 6 — чтение фронта** (@sec-pareto-front): наклоны компромиссов, связи и допустимые области попарных диаграмм; почему проекции четырёхмерного фронта нельзя соединять линиями (@sec-reading-scatter-matrix).
7. **Упражнение 7 — параллельные координаты** (@sec-parallel-coordinates): фильтрация ограничений, разброс параметров и цена сочетания целей.

Упражнения идут от физической интуиции (5) через формальное понимание пространств решений и критериев (D, E, F, G) к количественному анализу результатов (6, 7) и обсуждению управления (H).

# Приложение A: подробности настройки {#sec-appendix-config}

## Полный перечень параметров

Полный перечень дан в @tbl-full-knob-inventory и извлекается из `WaterSystem.param_schema()`, поэтому пособие соответствует реальному коду.

## Схема параметров

@tbl-reduced-config задаёт переменные и границы каждого семейства, имеющие физическое обоснование:

- **Доли экологического стока**: \[0, 1\], от только орошения (0) до только экологического направления (1).
- **SLOP `vr`**: \[0, 500\] m³/s, объявленная граница расхода выпуска.
- **`v1` и `v2`**: \[0, 10⁹\] m³. Ограничение: v1 ≤ v2.
- **`buffer_coef`**: \[0, 1\], полный диапазон коэффициентов уменьшения.
- **`flood_coef`**: \[0, 5\], множитель выпуска в паводковой зоне.

## Ограничения

**Ordered (v1 ≤ v2)**: оба водохранилища имеют `Ordered(low="v1", high="v2")`. Фабрика `make_repair()` taqsim создаёт функцию, меняющую v1 и v2 местами при нарушении после любой генетической операции.

Ограничения суммы долей, равной единице, не нужны: доли ветвей больше не являются переменными решения; используются единичные экологические доли или остатки, взвешенные по спросу.

## Гиперпараметры оптимизатора

| Параметр | Значение | Обоснование |
|:-----------------------|:-----------------------|:-----------------------|
| `pop_size` | 12 проверка / 300 демонстрация | Проверочный режим предотвращает случайные долгие расчёты; демонстрация даёт умеренный фронт |
| `n_generations` | 2 проверка / 100 демонстрация | Проверка только работоспособности; необходимость продления демонстрации определяется сходимостью |
| `eta` (SBX) | 15.0 | Стандартное значение для умеренного взаимодействия переменных |
| `eta` (мутация) | 20.0 | Стандартное значение для уточнения локального поиска |
| `n_workers` | 2 проверка / 6 демонстрация | Лёгкая проверка; умеренный параллелизм демонстрации |
| `seed` | 42 | Воспроизводимость |

## Логика функции исправления

```
taqsim.optimization.make_repair() автоматически создаёт:
1. clip(x, lower, upper)            # Границы переменных
2. Для каждого ограничения Ordered:
     if v1 > v2: swap(v1, v2)       # Ordered(v1 ≤ v2)
```

## Сочетание операторов

Выполняется внутри `taqsim.optimization.optimize()` через `run_optimization()`: скрещивание SBX, полиномиальная мутация и автоматически созданное исправление.

# Приложение B: словарь {#sec-glossary}

| Термин | Определение |
|:-----------------------|:----------------------------------------------|
| **Границы** | Минимум и максимум каждой переменной, определяющие допустимую область решений. |
| **Сходимость** | Стабилизация фронта по поколениям, измеряемая гиперобъёмом. |
| **Расстояние скученности** | Мера плотности: для разнообразия предпочтительны решения в менее плотных областях фронта. |
| **Переменная решения** | Настраиваемый оптимизатором параметр: здесь экологические доли распределителей и месячно-циклические значения SLOP. |
| **Доминирование** | A *доминирует* B, если A не хуже по всем критериям и строго лучше хотя бы по одному. |
| **Гиперобъём** | Объём пространства критериев, доминируемый фронтом относительно опорной точки. Больше — лучше. |
| **Точка перегиба** | Решение с наиболее резким изменением наклона компромисса: максимальное улучшение на единицу ухудшения. |
| **Недоминируемое решение** | Решение, не доминируемое никаким другим решением популяции. |
| **Целевая функция** | Функция, преобразующая конфигурацию параметров в скалярную оценку, например общий сельскохозяйственный дефицит. |
| **Фронт Парето** | Множество недоминируемых решений. Улучшение одного критерия обязательно ухудшает другой. |
| **Ранг Парето** | 0 — фронт Парето; 1 — фронт после удаления ранга 0; и так далее. |
| **Функция исправления** | Функция, возвращающая недопустимые решения в допустимую область после генетических операций. |

# Литература {.unnumbered}
